# Séance 6 — Sous le capot : Python avancé

## 🛟 Notebook « point de reprise »

**À quoi sert ce notebook ?** Si tu as manqué la séance précédente, si ton code
ne marche pas, ou si tu t'es perdu·e en route : **ouvre celui-ci**. Le code de
départ est déjà écrit et fonctionne. Tu n'as jamais besoin d'avoir réussi
l'exercice d'avant pour suivre celui d'aujourd'hui.

**Comment l'utiliser ?**
1. Exécute les cellules du haut sans les modifier (elles remettent tout en place).
2. Descends jusqu'aux cellules `# ✏️ À TOI DE JOUER`.
3. Écris ton code à la place des `...`.

**Raccourci** : `Maj + Entrée` exécute une cellule.

---


> 🎯 **Objectif réaliste pour cette séance** : savoir **lire** ce code.
> Savoir l'écrire viendra avec la pratique. Ne te décourage pas ici.


## 1. Les dunder methods = les prises normalisées

In [ ]:
class Carnet:
    def __init__(self, opportunites=None):
        self._items = opportunites or []

    def __len__(self):                 # active len(carnet)
        return len(self._items)

    def __iter__(self):                # active "for x in carnet"
        return iter(self._items)

    def __contains__(self, titre):     # active "titre in carnet"
        return any(i == titre for i in self._items)

    def __repr__(self):                # ce qui s'affiche dans une liste / le débogueur
        return f"Carnet({len(self._items)} opportunités)"


c = Carnet(["Bourse", "Stage", "Hackathon"])
print(len(c), "|", "Stage" in c, "|", c)
for item in c:
    print(" -", item)


## 2. Le générateur = le distributeur de tickets

In [ ]:
import sys

liste = list(range(10_000_000))
generateur = (x for x in range(10_000_000))

print("Liste      :", sys.getsizeof(liste), "octets")
print("Générateur :", sys.getsizeof(generateur), "octets")


In [ ]:
def compter_jusqua(n):
    """yield rend une valeur, MET EN PAUSE, puis reprend au tour suivant."""
    i = 0
    while i < n:
        yield i
        i += 1


for valeur in compter_jusqua(5):
    print(valeur, end=" ")


## 3. Le décorateur — les 4 marches

Ne saute aucune marche : c'est le seul chemin qui mène à la compréhension.


In [ ]:
# MARCHE 1 : une fonction est une valeur comme une autre.
def dire_bonjour():
    print("Bonjour !")

f = dire_bonjour      # SANS parenthèses : on copie la fonction, on ne l'appelle pas
f()


In [ ]:
# MARCHE 2 : une fonction peut RECEVOIR une fonction.
def deux_fois(fonction):
    fonction()
    fonction()

deux_fois(dire_bonjour)


In [ ]:
# MARCHE 3 : une fonction peut FABRIQUER une fonction. La marche difficile.
def crier(fonction):
    def enveloppe():
        print(">>> AVANT <<<")
        fonction()
        print(">>> APRÈS <<<")
    return enveloppe          # on rend la fonction, pas son résultat

bruyant = crier(dire_bonjour)
bruyant()


In [ ]:
# MARCHE 4 : @ est exactement la même chose, en plus court.
@crier
def dire_bonsoir():
    print("Bonsoir !")

dire_bonsoir()
# équivaut à : dire_bonsoir = crier(dire_bonsoir)


## 4. Le décorateur utile : `@chronometre`

In [ ]:
import functools
import time


def chronometre(fonction):
    @functools.wraps(fonction)     # conserve le nom et la docstring
    def enveloppe(*args, **kwargs):
        depart = time.perf_counter()
        resultat = fonction(*args, **kwargs)
        duree = time.perf_counter() - depart
        print(f"[chrono] {fonction.__name__} : {duree:.3f} s")
        return resultat            # ⚠️ SANS ce return, tout casse en aval
    return enveloppe


@chronometre
def charger_beaucoup(n: int) -> list[int]:
    """Simule un traitement lent."""
    return [i ** 2 for i in range(n)]


_ = charger_beaucoup(3_000_000)
print("Nom conservé :", charger_beaucoup.__name__)


## 5. `match` structurel — bien au-delà du menu

In [ ]:
evenement = {"type": "offre", "pays": "Maroc", "jours": 3}

match evenement:
    case {"type": "offre", "jours": int(j)} if j < 7:
        print("Offre urgente :", j, "jours")
    case {"type": "offre"}:
        print("Offre normale")
    case _:
        print("Inconnu")


---
# ✏️ À TOI DE JOUER

1. Écris `@journalise` : affiche le nom de la fonction et ses arguments avant l'appel.
2. **Bonus difficile** : `@reessayer(n=3)`, un décorateur *paramétré*
   (trois niveaux d'imbrication). Il servira tel quel en séance 9.


In [ ]:
# ✏️ À TOI DE JOUER
import functools


def journalise(fonction):
    @functools.wraps(fonction)
    def enveloppe(*args, **kwargs):
        ...
    return enveloppe


## 🧪 Atelier qualité — tes 3 premiers tests

In [ ]:
# En Colab, pytest s'utilise via un fichier. En local :
#   uv run pytest   (ou simplement : pytest)
# Le corrigé est dans fil-rouge/v3-avance/tests/

def est_urgente(jours: int) -> bool:
    return 0 <= jours < 7


# Version "à la main" pour comprendre ce que fait assert :
assert est_urgente(3) is True
assert est_urgente(7) is False
assert est_urgente(-1) is False
print("✅ Les 3 assertions passent")
